# Отчёт по рекламе Ozon с юнит-экономикой

Ноутбук читает данные из **Ozon Performance API** (реклама) и — при наличии ключей — из **Ozon Seller API** (воронка продаж, цены, комиссии) и собирает Excel в формате привычного отчёта Wildberries: **Сводка / Кампании / Товары / Ключи / Воронка / Экономика**.

Ноутбук ничего не меняет в кабинете: не запускает и не отключает кампании, не трогает бюджеты, ставки и цены.

## Что отвечает отчёт
1. **Эффективна ли кампания** — лист «Кампании»: ДРР, CPO, CTR, вердикт по целевому ДРР.
2. **Сходится ли реклама с экономикой** — листы «Экономика» и «Кампании»: маржа до рекламы по каждому SKU, **предельный ДРР** (при котором кампания крутится в ноль), запас в процентных пунктах и расчётная прибыль с рекламы.

## Перед первым запуском
1. В кабинете Ozon Seller откройте **Настройки → API-ключи → Performance API** и создайте сервисный аккаунт.
2. В Colab слева откройте **Секреты** (иконка ключа) и добавьте `OZON_PERF_CLIENT_ID` и `OZON_PERF_CLIENT_SECRET`.
3. *(Для воронки и экономики)* Там же в кабинете создайте обычный ключ **Seller API** и добавьте секреты `OZON_SELLER_CLIENT_ID` (Client-Id, число) и `OZON_SELLER_API_KEY`.
4. Включите для всех секретов доступ из блокнота.
5. *(Для точной экономики)* Загрузите в Colab (панель слева → Файлы) файл `cost_prices.csv` с себестоимостью: колонки `артикул;себестоимость` (или `sku;себестоимость`). Без файла себестоимость берётся как доля цены из параметра `DEFAULT_COST_SHARE_PCT`.

## Запуск
1. Укажите период и параметры экономики в ячейке **Параметры**.
2. Выберите **Среда выполнения → Выполнить все**.
3. В конце Colab скачает сформированный `.xlsx`.

> Асинхронные отчёты Ozon формируются последовательно из-за лимита на одновременные выгрузки — выгрузка может занять несколько минут.


In [ ]:
#@title Параметры { display-mode: "form" }
NOTEBOOK_VERSION = "2026-08-31-econ-v5"
DATE_FROM = "2026-08-28"  #@param {type:"string"}
DATE_TO = "2026-08-30"    #@param {type:"string"}
TARGET_DRR = 10.0            #@param {type:"number"}
NO_ORDERS_SPEND_LIMIT = 1000.0  #@param {type:"number"}
MIN_CLICKS_FOR_VERDICT = 20  #@param {type:"integer"}
INCLUDE_CPO = True           #@param {type:"boolean"}
INCLUDE_PHRASES = True       #@param {type:"boolean"}
INCLUDE_CURRENT_BIDS = True  #@param {type:"boolean"}
INCLUDE_ALL_SKU_CPO = True   #@param {type:"boolean"}

#@markdown ---
#@markdown **Экономика и предельный ДРР** (нужны секреты Seller API)
INCLUDE_ECONOMICS = True     #@param {type:"boolean"}
INCLUDE_FUNNEL = True        #@param {type:"boolean"}
SALES_SCHEME = "FBO"         #@param ["FBO", "FBS"]
TAX_RATE_PCT = 6.0           #@param {type:"number"}
DEFAULT_COST_SHARE_PCT = 30.0  #@param {type:"number"}
EXTRA_COST_PER_UNIT = 0.0    #@param {type:"number"}
BREAKEVEN_SAFETY_PP = 2.0    #@param {type:"number"}
COST_FILE = "cost_prices.csv"  #@param {type:"string"}
#@markdown `TAX_RATE_PCT` — налог с выручки (УСН «доходы» = 6, самозанятость = 4–6, 0 — не учитывать).
#@markdown `DEFAULT_COST_SHARE_PCT` — себестоимость как % от цены для SKU, которых нет в файле себестоимости (0 — не подставлять допущение).
#@markdown `EXTRA_COST_PER_UNIT` — упаковка, обработка и прочие затраты на штуку, ₽.
#@markdown `BREAKEVEN_SAFETY_PP` — коридор «около нуля» вокруг предельного ДРР, п.п.

from google.colab import userdata

def colab_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

OZON_PERF_CLIENT_ID = colab_secret('OZON_PERF_CLIENT_ID')
OZON_PERF_CLIENT_SECRET = colab_secret('OZON_PERF_CLIENT_SECRET')
if not OZON_PERF_CLIENT_ID or not OZON_PERF_CLIENT_SECRET:
    raise ValueError('Добавьте секреты OZON_PERF_CLIENT_ID и OZON_PERF_CLIENT_SECRET и включите доступ из блокнота.')

OZON_SELLER_CLIENT_ID = colab_secret('OZON_SELLER_CLIENT_ID')
OZON_SELLER_API_KEY = colab_secret('OZON_SELLER_API_KEY')
SELLER_API_AVAILABLE = bool(OZON_SELLER_CLIENT_ID and OZON_SELLER_API_KEY)

print(f'Ноутбук {NOTEBOOK_VERSION}. Реквизиты Performance API получены из секретов Colab.')
if SELLER_API_AVAILABLE:
    print('Seller API подключён: будут собраны листы «Воронка» и «Экономика».')
else:
    print('Секреты OZON_SELLER_CLIENT_ID / OZON_SELLER_API_KEY не заданы — листы «Воронка» и «Экономика» будут пропущены.')


## Клиенты API и служебные функции
Эту ячейку менять не требуется.

In [ ]:
import datetime as dt
import io
import json
import re
import time
import zipfile
from urllib.parse import urljoin

import pandas as pd
import requests

API_URL = 'https://api-performance.ozon.ru'

def chunks(values, size):
    values = list(values)
    for i in range(0, len(values), size):
        yield values[i:i + size]

def rows_from(data, preferred=('rows', 'list', 'products', 'items', 'data', 'result')):
    if data is None:
        return []
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in preferred:
            value = data.get(key)
            if isinstance(value, list):
                return value
            if isinstance(value, dict):
                nested = rows_from(value, preferred)
                if nested:
                    return nested
        return [data]
    return []

def decode_ozon_text(raw):
    # Исторические CSV Ozon встречаются как в UTF-8, так и в Windows-1251.
    for encoding in ('utf-8-sig', 'cp1251'):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            pass
    return raw.decode('utf-8-sig', errors='replace')

def unwrap_ozon_csv_line(line):
    # В официальных выгрузках Ozon вся строка с разделителями иногда обёрнута в кавычки.
    stripped = line.strip()
    if len(stripped) >= 2 and stripped[0] == stripped[-1] == '\"' and ';' in stripped:
        return stripped[1:-1].replace('\"\"', '\"')
    return line

def parse_ozon_csv(text, source_name=''):
    lines = text.splitlines()
    if not lines:
        return pd.DataFrame()
    header_index = None
    for index, line in enumerate(lines):
        candidate = unwrap_ozon_csv_line(line)
        normalized = candidate.casefold().replace('ё', 'е')
        markers = ('sku', 'дата', 'период', 'название', 'наименование', 'идентификатор', 'показы', 'расход', 'заказы', 'продажи')
        if ';' in candidate and sum(marker in normalized for marker in markers) >= 2:
            header_index = index
            break
    if header_index is None:
        preview = ' | '.join(lines[:3])[:500]
        raise ValueError(f'Не найден заголовок CSV Ozon ({source_name or "без имени"}): {preview}')
    preamble = '\n'.join(lines[:header_index])
    report_lines = [unwrap_ozon_csv_line(line) for line in lines[header_index:] if line.strip()]
    report_text = '\n'.join(report_lines)
    separator = ';' if ';' in report_lines[0] else None
    frame = pd.read_csv(io.StringIO(report_text), sep=separator, engine='python', dtype=str)
    frame.columns = [str(column).strip() for column in frame.columns]
    campaign_match = re.search(r'№\s*(\d+)', preamble)
    if campaign_match and 'campaignId' not in frame.columns:
        frame['campaignId'] = campaign_match.group(1)
    if source_name:
        frame['_source_file'] = source_name
    return frame

def response_to_df(response):
    content_type = (response.headers.get('Content-Type') or '').lower()
    raw = response.content
    if 'zip' in content_type or raw[:2] == b'PK':
        frames = []
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            for name in archive.namelist():
                if name.startswith('__MACOSX/') or name.endswith('/'):
                    continue
                if name.lower().endswith(('.csv', '.txt')):
                    text = decode_ozon_text(archive.read(name))
                    part = parse_ozon_csv(text, name)
                    frames.append(part)
                elif name.lower().endswith('.json'):
                    data = json.loads(archive.read(name).decode('utf-8-sig'))
                    part = pd.json_normalize(rows_from(data))
                    part['_source_file'] = name
                    frames.append(part)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    text = decode_ozon_text(raw)
    if 'json' in content_type or text.lstrip().startswith(('{', '[')):
        data = json.loads(text)
        return pd.json_normalize(rows_from(data))
    if not text.strip():
        return pd.DataFrame()
    return parse_ozon_csv(text)

class OzonPerformance:
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.session = requests.Session()
        self.token = None
        self.token_expires_at = 0

    def refresh_token(self):
        response = self.session.post(
            f'{API_URL}/api/client/token',
            json={'client_id': self.client_id, 'client_secret': self.client_secret, 'grant_type': 'client_credentials'},
            headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
            timeout=60,
        )
        if not response.ok:
            raise RuntimeError(f'Не удалось получить токен: {response.status_code} {response.text[:500]}')
        data = response.json()
        self.token = data['access_token']
        self.token_expires_at = time.time() + int(data.get('expires_in', 1800)) - 60
        self.session.headers.update({'Authorization': f'Bearer {self.token}', 'Accept': 'application/json'})

    def request(self, method, path, max_retries=8, **kwargs):
        url = path if path.startswith('http') else urljoin(API_URL + '/', path.lstrip('/'))
        last_problem = ''
        for attempt in range(max_retries):
            if not self.token or time.time() >= self.token_expires_at:
                self.refresh_token()
            try:
                response = self.session.request(method, url, timeout=180, **kwargs)
            except requests.RequestException as exc:
                last_problem = f'{type(exc).__name__}: {exc}'
                wait = min(10 * (attempt + 1), 60)
                print(f'  Сетевая ошибка, повтор через {wait} сек. ({attempt + 1}/{max_retries})')
                time.sleep(wait)
                continue
            if response.status_code == 401:
                self.refresh_token()
                continue
            if response.status_code in (408, 409, 425, 429) or response.status_code >= 500:
                last_problem = f'{response.status_code}: {response.text[:700]}'
                retry_header = response.headers.get('Retry-After')
                try:
                    wait = int(float(retry_header)) if retry_header else min(15 * (attempt + 1), 60)
                except (TypeError, ValueError):
                    wait = min(15 * (attempt + 1), 60)
                wait = max(3, min(wait, 120))
                print(f'  API временно занят ({response.status_code}), повтор через {wait} сек. ({attempt + 1}/{max_retries})')
                time.sleep(wait)
                continue
            if not response.ok:
                raise RuntimeError(f'{method} {url} -> {response.status_code}: {response.text[:700]}')
            return response
        raise RuntimeError(f'{method} {url}: исчерпаны повторные попытки. Последний ответ: {last_problem}')

    def json(self, method, path, **kwargs):
        response = self.request(method, path, **kwargs)
        return response.json() if response.content else {}

    def dataframe(self, method, path, **kwargs):
        return response_to_df(self.request(method, path, **kwargs))

    def report(self, method, path, *, params=None, body=None, timeout_sec=900):
        # Создание отчёта иногда получает 429/409, пока Ozon завершает предыдущую выгрузку.
        response = self.request(method, path, params=params, json=body, max_retries=12)
        data = response.json()
        report_id = data.get('UUID') or data.get('uuid')
        if not report_id:
            raise RuntimeError(f'API не вернул UUID отчёта: {data}')
        started = time.time()
        pause = 3
        while True:
            status = self.json('GET', f'/api/client/statistics/{report_id}')
            state = status.get('state')
            if state == 'OK':
                link = status.get('link')
                if link:
                    return self.dataframe('GET', link)
                return self.dataframe('GET', '/api/client/statistics/report', params={'UUID': report_id})
            if state == 'ERROR':
                raise RuntimeError(status.get('error') or f'Ошибка формирования отчёта {report_id}')
            if time.time() - started > timeout_sec:
                raise TimeoutError(f'Отчёт {report_id} не сформировался за {timeout_sec} сек.')
            time.sleep(pause)
            pause = min(pause + 1, 10)

SELLER_API_URL = 'https://api-seller.ozon.ru'

class OzonSeller:
    """Клиент Ozon Seller API: авторизация по заголовкам Client-Id и Api-Key."""

    def __init__(self, client_id, api_key):
        self.session = requests.Session()
        self.session.headers.update({
            'Client-Id': str(client_id),
            'Api-Key': str(api_key),
            'Content-Type': 'application/json',
            'Accept': 'application/json',
        })

    def json(self, method, path, body=None, max_retries=8):
        url = urljoin(SELLER_API_URL + '/', path.lstrip('/'))
        last_problem = ''
        for attempt in range(max_retries):
            try:
                response = self.session.request(method, url, json=body, timeout=180)
            except requests.RequestException as exc:
                last_problem = f'{type(exc).__name__}: {exc}'
                wait = min(10 * (attempt + 1), 60)
                print(f'  Сетевая ошибка Seller API, повтор через {wait} сек. ({attempt + 1}/{max_retries})')
                time.sleep(wait)
                continue
            if response.status_code in (408, 425, 429) or response.status_code >= 500:
                last_problem = f'{response.status_code}: {response.text[:500]}'
                retry_header = response.headers.get('Retry-After')
                try:
                    wait = int(float(retry_header)) if retry_header else min(15 * (attempt + 1), 60)
                except (TypeError, ValueError):
                    wait = min(15 * (attempt + 1), 60)
                wait = max(3, min(wait, 120))
                print(f'  Seller API временно занят ({response.status_code}), повтор через {wait} сек. ({attempt + 1}/{max_retries})')
                time.sleep(wait)
                continue
            if not response.ok:
                raise RuntimeError(f'{method} {url} -> {response.status_code}: {response.text[:700]}')
            return response.json() if response.content else {}
        raise RuntimeError(f'{method} {url}: исчерпаны повторные попытки. Последний ответ: {last_problem}')

print('Клиенты API загружены.')

## Загрузка данных
Ошибки отдельных необязательных разделов записываются на лист «Диагностика», а остальные части отчёта продолжают формироваться.

In [ ]:
def rfc3339_start(date_text):
    return f'{date_text}T00:00:00Z'

def rfc3339_end(date_text):
    return f'{date_text}T23:59:59Z'

def load_campaigns(api):
    print('1/7 Список кампаний...')
    data = api.json('GET', '/api/client/campaign')
    frame = pd.json_normalize(rows_from(data, ('list', 'campaigns', 'items', 'data')))
    print('  кампаний:', len(frame))
    return frame

def campaign_ids_by_type(campaigns, object_type):
    if campaigns.empty or 'advObjectType' not in campaigns.columns:
        return []
    mask = campaigns['advObjectType'].astype(str).str.upper().eq(object_type)
    return campaigns.loc[mask, 'id'].astype(str).str.replace(r'\.0$', '', regex=True).tolist()

def load_cpc_campaign_stats(api, campaign_ids, d_from, d_to):
    print('2/7 Итоги CPC-кампаний...')
    frames = []
    for batch in chunks(campaign_ids, 10):
        params = [('dateFrom', d_from), ('dateTo', d_to)] + [('campaignIds', value) for value in batch]
        frames.append(api.dataframe('GET', '/api/client/statistics/campaign/product/json', params=params))
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print('  строк:', len(result))
    return result

def load_daily_stats(api, campaign_ids, d_from, d_to):
    frames = []
    for batch in chunks(campaign_ids, 10):
        params = [('dateFrom', d_from), ('dateTo', d_to)] + [('campaignIds', value) for value in batch]
        frames.append(api.dataframe('GET', '/api/client/statistics/daily/json', params=params))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_expense_stats(api, campaign_ids, d_from, d_to):
    print('  Расходы по источникам...')
    frames = []
    for batch in chunks(campaign_ids, 10):
        params = [('dateFrom', d_from), ('dateTo', d_to)] + [('campaignIds', value) for value in batch]
        frames.append(api.dataframe('GET', '/api/client/statistics/expense/json', params=params))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_media_campaign_stats(api, campaign_ids, d_from, d_to):
    print('  Статистика медийных кампаний...')
    frames = []
    for batch in chunks(campaign_ids, 10):
        params = [('dateFrom', d_from), ('dateTo', d_to)] + [('campaignIds', value) for value in batch]
        frames.append(api.dataframe('GET', '/api/client/statistics/campaign/media/json', params=params))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_video_details(api, campaign_ids, d_from, d_to):
    print('  Детальная статистика видеобаннеров...')
    frames = []
    for batch in chunks(campaign_ids, 10):
        frames.append(api.report('POST', '/api/client/statistics/video/json', body={
            'campaigns': batch, 'dateFrom': d_from, 'dateTo': d_to, 'groupBy': 'NO_GROUP_BY'
        }))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_cpc_sku_stats(api, campaign_ids, d_from, d_to):
    print('3/7 Историческая статистика CPC по SKU...')
    frames = []
    for batch in chunks(campaign_ids, 10):
        frames.append(api.report('POST', '/api/client/statistics', body={
            'campaigns': batch, 'dateFrom': d_from, 'dateTo': d_to, 'groupBy': 'NO_GROUP_BY'
        }))
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print('  строк:', len(result))
    return result

def load_cpc_products(api, campaign_ids, competitive=True):
    print('4/7 Товары и текущие настройки CPC...')
    product_rows, competitive_rows = [], []
    for number, campaign_id in enumerate(campaign_ids, 1):
        page, campaign_products = 1, []
        while True:
            data = api.json('GET', f'/api/client/campaign/{campaign_id}/v2/products', params={'page': page, 'pageSize': 1000})
            rows = rows_from(data, ('products', 'items', 'list', 'data'))
            if not rows:
                break
            for row in rows:
                row = dict(row)
                row['campaignId'] = campaign_id
                product_rows.append(row)
                campaign_products.append(row)
            if len(rows) < 1000:
                break
            page += 1
        if competitive:
            skus = [str(row.get('sku')) for row in campaign_products if row.get('sku')]
            for batch in chunks(skus, 200):
                params = [('skus', sku) for sku in batch]
                data = api.json('GET', f'/api/client/campaign/{campaign_id}/products/bids/competitive', params=params)
                for row in rows_from(data, ('bids', 'items', 'data')):
                    row = dict(row)
                    row['campaignId'] = campaign_id
                    competitive_rows.append(row)
        if number % 20 == 0:
            print(f'  обработано кампаний: {number}/{len(campaign_ids)}')
    print('  товаров:', len(product_rows))
    return pd.json_normalize(product_rows), pd.json_normalize(competitive_rows)

def load_cpo_products(api):
    print('5/7 Товары CPO...')
    rows, page = [], 1
    while True:
        data = api.json('POST', '/api/client/campaign/search_promo/v2/products', json={'page': page, 'pageSize': 1000})
        part = rows_from(data, ('products', 'items', 'data'))
        if not part:
            break
        rows.extend(part)
        if len(part) < 1000:
            break
        page += 1
    frame = pd.json_normalize(rows)
    if not frame.empty and 'sku' in frame.columns:
        minimums = []
        for batch in chunks(frame['sku'].dropna().astype(str), 200):
            data = api.json('POST', '/api/client/search_promo/get_cpo_min_bids', json={'skus': list(batch)})
            minimums.extend(rows_from(data, ('bids', 'items', 'data')))
        min_frame = pd.json_normalize(minimums)
        if not min_frame.empty:
            min_frame = min_frame.rename(columns={'bid': 'minBid'})
            frame['sku'] = frame['sku'].astype(str).str.replace(r'\.0$', '', regex=True)
            min_frame['sku'] = min_frame['sku'].astype(str).str.replace(r'\.0$', '', regex=True)
            frame = frame.merge(min_frame[['sku', 'minBid']], on='sku', how='left')
    print('  товаров:', len(frame))
    return frame

def load_cpo_statistics(api, d_from, d_to):
    print('6/7 Асинхронный отчёт CPO по товарам...')
    return api.report('POST', '/api/client/statistic/products/generate/json', body={
        'from': rfc3339_start(d_from), 'to': rfc3339_end(d_to)
    })

def load_cpo_orders(api, d_from, d_to):
    return api.report('POST', '/api/client/statistic/orders/generate/json', body={
        'from': rfc3339_start(d_from), 'to': rfc3339_end(d_to)
    })

def load_all_sku_cpo_statistics(api, d_from, d_to):
    print('  Отчёт CPO по всем товарам (Plus)...')
    try:
        return api.report('GET', '/api/client/statistics/all_sku_promo/products/generate/json', params={
            'timeBounds.from': rfc3339_start(d_from), 'timeBounds.to': rfc3339_end(d_to)
        })
    except RuntimeError as exc:
        message = str(exc)
        if '404' in message and 'Кампания не найдена' in message:
            print('  Кампания «Оплата за заказ — все товары» не найдена; раздел Plus пропущен.')
            return pd.DataFrame()
        raise

def load_all_sku_cpo_orders(api, d_from, d_to):
    return api.report('GET', '/api/client/statistics/all_sku_promo/orders/generate/json', params={
        'timeBounds.from': rfc3339_start(d_from), 'timeBounds.to': rfc3339_end(d_to)
    })

def load_phrases(api, campaigns, d_from, d_to):
    print('7/7 Поисковые фразы...')
    if campaigns.empty:
        return pd.DataFrame()
    placement = campaigns.get('placement', pd.Series('', index=campaigns.index)).astype(str)
    object_type = campaigns.get('advObjectType', pd.Series('', index=campaigns.index)).astype(str)
    ids = campaigns.loc[object_type.eq('SKU') & placement.str.contains('PLACEMENT_TOP_PROMOTION', na=False), 'id']
    ids = ids.astype(str).str.replace(r'\.0$', '', regex=True).tolist()
    frames = []
    for batch in chunks(ids, 10):
        frames.append(api.report('POST', '/api/client/statistics/phrases/json', body={
            'campaigns': batch, 'dateFrom': d_from, 'dateTo': d_to, 'groupBy': 'NO_GROUP_BY'
        }))
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print('  строк:', len(result))
    return result

def load_seller_products(seller):
    """Каталог продавца: связка SKU <-> артикул <-> название (v3/product/list + v3/product/info/list)."""
    print('Seller API: каталог товаров...')
    ids, last_id = [], ''
    while True:
        data = seller.json('POST', '/v3/product/list', body={
            'filter': {'visibility': 'ALL'}, 'last_id': last_id, 'limit': 1000,
        })
        result = data.get('result') or {}
        items = result.get('items') or []
        ids.extend(str(item.get('product_id')) for item in items if item.get('product_id'))
        last_id = result.get('last_id') or ''
        if len(items) < 1000 or not last_id:
            break
    rows = []
    for batch in chunks(ids, 1000):
        data = seller.json('POST', '/v3/product/info/list', body={'product_id': list(batch)})
        for item in rows_from(data, ('items', 'result')):
            rows.append({
                'product_id': str(item.get('id') or ''),
                'sku': str(item.get('sku') or ''),
                'offer_id': item.get('offer_id') or '',
                'name': item.get('name') or '',
            })
    frame = pd.DataFrame(rows)
    print('  товаров в каталоге:', len(frame))
    return frame

def load_seller_prices(seller):
    """Цены, комиссии, логистика и эквайринг по каждому товару (v5/product/info/prices)."""
    print('Seller API: цены и комиссии...')
    rows, cursor = [], ''
    while True:
        data = seller.json('POST', '/v5/product/info/prices', body={
            'filter': {'visibility': 'ALL'}, 'cursor': cursor, 'limit': 1000,
        })
        items = data.get('items') or rows_from(data, ('items',))
        if not items:
            break
        rows.extend(items)
        cursor = data.get('cursor') or ''
        if len(items) < 1000 or not cursor:
            break
    frame = pd.json_normalize(rows)
    print('  строк с ценами:', len(frame))
    return frame

FUNNEL_METRICS_FULL = [
    'revenue', 'ordered_units', 'session_view_pdp', 'hits_view_pdp', 'hits_tocart',
    'conv_tocart_pdp', 'position_category', 'delivered_units', 'returns', 'cancellations',
]
FUNNEL_METRICS_BASIC = ['revenue', 'ordered_units']

def load_funnel(seller, d_from, d_to, max_rows=10000):
    """Воронка продаж по SKU (v1/analytics/data). Полный набор метрик доступен на Premium Plus;
    без подписки автоматически откатываемся на revenue + ordered_units."""
    print('Seller API: воронка продаж по SKU...')
    for metrics in (FUNNEL_METRICS_FULL, FUNNEL_METRICS_BASIC):
        rows, offset = [], 0
        try:
            while True:
                data = seller.json('POST', '/v1/analytics/data', body={
                    'date_from': d_from, 'date_to': d_to, 'dimension': ['sku'],
                    'metrics': metrics, 'limit': 1000, 'offset': offset,
                })
                part = (data.get('result') or {}).get('data') or []
                for entry in part:
                    dims = entry.get('dimensions') or [{}]
                    row = {'sku': str(dims[0].get('id') or ''), 'name': dims[0].get('name') or ''}
                    for metric, value in zip(metrics, entry.get('metrics') or []):
                        row[metric] = value
                    rows.append(row)
                offset += 1000
                if len(part) < 1000 or offset >= max_rows:
                    break
                time.sleep(1)
        except RuntimeError as exc:
            if metrics is FUNNEL_METRICS_FULL:
                print(f'  Полный набор метрик недоступен ({str(exc)[:120]}...), пробуем базовый.')
                continue
            raise
        print(f'  строк воронки: {len(rows)}, метрик: {len(metrics)}')
        return pd.DataFrame(rows), metrics
    return pd.DataFrame(), []

print('Функции загрузки готовы.')

## Расчёты и Excel
Названия полей нормализуются как для JSON, так и для русскоязычных CSV-выгрузок Ozon.

In [ ]:
def key_name(value):
    return re.sub(r'[^a-zа-я0-9]+', '', str(value).casefold().replace('ё', 'е'))

SCHEMA_WARNINGS = []

def get_col(frame, aliases, default=''):
    if frame.empty:
        return pd.Series(dtype='object')
    lookup = {key_name(column): column for column in frame.columns}
    for alias in aliases:
        column = lookup.get(key_name(alias))
        if column is not None:
            return frame[column]
    return pd.Series(default, index=frame.index)

def require_col(frame, aliases, label):
    lookup = {key_name(column): column for column in frame.columns}
    for alias in aliases:
        column = lookup.get(key_name(alias))
        if column is not None:
            return frame[column]
    available = ', '.join(map(str, frame.columns))
    message = f'Не найдено поле «{label}». Получены колонки: {available}'
    print('  Внимание —', message)
    SCHEMA_WARNINGS.append({'Раздел': f'Схема данных: {label}', 'Ошибка': message})
    return pd.Series(pd.NA, index=frame.index, dtype='object')

def number(series):
    if series.empty:
        return pd.Series(dtype='float64')
    cleaned = (series.astype(str).str.replace('\u00a0', '', regex=False).str.replace(' ', '', regex=False)
               .str.replace(',', '.', regex=False).str.replace(r'[^0-9.\-]', '', regex=True))
    return pd.to_numeric(cleaned, errors='coerce').fillna(0.0)

def text_id(series):
    return series.astype(str).str.replace(r'\.0$', '', regex=True).replace({'nan': '', 'None': ''})

def pct(a, b):
    return round(a / b * 100, 1) if b else 0.0

def verdict(spend, sales, orders, clicks, target_drr):
    if spend <= 0:
        return 'Нет затрат'
    if clicks < MIN_CLICKS_FOR_VERDICT and orders == 0:
        return 'Мало данных'
    if orders == 0:
        return 'Отключить: расход без заказов' if spend >= NO_ORDERS_SPEND_LIMIT else 'Мало данных'
    if sales <= 0:
        return 'Мало данных'
    drr = spend / sales * 100
    if drr <= target_drr:
        return 'Эффективна / масштабировать'
    if drr <= target_drr * 2:
        return 'Наблюдать / оптимизировать'
    return 'Неэффективна / снизить расход'

def campaign_table(campaigns, stats):
    base = pd.DataFrame(index=campaigns.index)
    base['ID'] = text_id(get_col(campaigns, ['id', 'campaignId', 'Идентификатор кампании']))
    base['Название'] = get_col(campaigns, ['title', 'name', 'Название кампании'])
    base['Статус'] = get_col(campaigns, ['state', 'Статус кампании'])
    base['Тип'] = get_col(campaigns, ['advObjectType', 'Тип объекта'])
    base['Модель оплаты'] = get_col(campaigns, ['paymentType', 'Тип продвижения'])
    base['Размещение'] = get_col(campaigns, ['placement', 'Места размещения'])
    base['Стратегия'] = get_col(campaigns, ['productAutopilotStrategy', 'Стратегия'])
    base['Бюджет, ₽'] = number(get_col(campaigns, ['budget'])) / 1_000_000
    base['Недельный бюджет, ₽'] = number(get_col(campaigns, ['weeklyBudget'])) / 1_000_000
    base['Дневной бюджет, ₽'] = number(get_col(campaigns, ['dailyBudget'])) / 1_000_000
    base['Дата начала'] = get_col(campaigns, ['fromDate'])
    base['Дата окончания'] = get_col(campaigns, ['toDate'])
    base['Создана'] = get_col(campaigns, ['createdAt'])
    base['Обновлена'] = get_col(campaigns, ['updatedAt'])

    stats_data = stats.copy()
    if not stats_data.empty:
        sku_values = text_id(get_col(stats_data, ['sku', 'SKU']))
        stats_data = stats_data[~sku_values.str.casefold().isin(['всего', 'итого', 'total'])]
    metrics = pd.DataFrame(index=stats_data.index)
    metrics['ID'] = text_id(get_col(stats_data, ['campaignId', 'id', 'Идентификатор кампании', 'ID кампании']))
    for target, aliases in {
        'Показы': ['views', 'Показы'], 'Клики': ['clicks', 'Клики'],
        'Корзины': ['toCart', 'Добавления в корзину', 'В корзину'],
        'Заказы': ['orders', 'Заказы', 'Заказы в штуках', 'Продано товаров'],
        'Продажи, ₽': ['sales', 'Заказы в рублях', 'Продажи', 'Продажи, ₽', 'Продажи в продвижении', 'Продажи в продвижении, ₽', 'Стоимость заказов', 'Выручка'],
        'Заказы модели': ['modelOrders', 'Заказы модели', 'Продано товаров модели'],
        'Продажи модели, ₽': ['modelSales', 'Продажи с заказов модели', 'Продажи в продвижении с заказов модели', 'Продажи в продвижении с заказов модели, ₽'],
        'Заказано всего, ₽': ['orderedSum', 'Заказано на сумму', 'Заказано на сумму, ₽'],
        'Расход, ₽': ['expense', 'Расход', 'Расход, ₽', 'Расход, ₽, с НДС', 'Затраты']}.items():
        metrics[target] = number(get_col(stats_data, aliases))
    if not metrics.empty:
        metric_columns = ['Показы', 'Клики', 'Корзины', 'Заказы', 'Продажи, ₽', 'Заказы модели', 'Продажи модели, ₽', 'Заказано всего, ₽', 'Расход, ₽']
        metrics = metrics.groupby('ID', as_index=False)[metric_columns].sum()
    result = base.merge(metrics, on='ID', how='left') if not metrics.empty else base
    for column in ['Показы', 'Клики', 'Корзины', 'Заказы', 'Продажи, ₽', 'Заказы модели', 'Продажи модели, ₽', 'Заказано всего, ₽', 'Расход, ₽']:
        if column not in result:
            result[column] = 0.0
        result[column] = pd.to_numeric(result[column], errors='coerce').fillna(0)
    result['CTR %'] = [pct(c, v) for c, v in zip(result['Клики'], result['Показы'])]
    result['CPC, ₽'] = [round(s / c, 2) if c else 0 for s, c in zip(result['Расход, ₽'], result['Клики'])]
    result['CR клик→заказ %'] = [pct(o, c) for o, c in zip(result['Заказы'], result['Клики'])]
    result['CPO, ₽'] = [round(s / o, 2) if o else 0 for s, o in zip(result['Расход, ₽'], result['Заказы'])]
    result['ДРР %'] = [pct(s, r) for s, r in zip(result['Расход, ₽'], result['Продажи, ₽'])]
    result['ДРР модели %'] = [pct(s, r) for s, r in zip(result['Расход, ₽'], result['Продажи модели, ₽'])]
    result['ДРР общий %'] = [pct(s, r) for s, r in zip(result['Расход, ₽'], result['Заказано всего, ₽'])]
    result['Вердикт'] = [verdict(s, r, o, c, TARGET_DRR) for s, r, o, c in zip(
        result['Расход, ₽'], result['Продажи, ₽'], result['Заказы'], result['Клики'])]
    return result.sort_values('Расход, ₽', ascending=False)

def cpc_product_table(stats, products, competitive, campaigns):
    if stats.empty:
        return pd.DataFrame()
    data = pd.DataFrame(index=stats.index)
    data['Кампания ID'] = text_id(get_col(stats, ['campaignId', 'ID кампании']))
    data['SKU'] = text_id(require_col(stats, ['sku', 'SKU'], 'SKU'))
    data['Дата'] = get_col(stats, ['date', 'Дата'])
    data['Дата добавления'] = get_col(stats, ['dateAdded', 'Дата добавления'])
    data['Товар из отчёта'] = get_col(stats, ['title', 'Название товара', 'Наименование'])
    for target, aliases in {
        'Показы': ['views', 'Показы'], 'Клики': ['clicks', 'Клики'],
        'Корзины': ['toCart', 'В корзину', 'Добавления в корзину'],
        'Заказы': ['orders', 'Заказы', 'Продано товаров'],
        'Продажи, ₽': ['sales', 'Продажи', 'Продажи в продвижении', 'Продажи в продвижении, ₽', 'Стоимость заказов', 'Выручка', 'Выручка, ₽', 'Выручка, Р'],
        'Заказы модели': ['modelOrders', 'Заказы модели', 'Продано товаров модели'],
        'Продажи модели, ₽': ['modelSales', 'Продажи с заказов модели', 'Продажи в продвижении с заказов модели', 'Продажи в продвижении с заказов модели, ₽'],
        'Заказано всего, ₽': ['orderedSum', 'Заказано на сумму', 'Заказано на сумму, ₽'],
        'Средний CPC Ozon, ₽': ['avgCpc', 'Средняя стоимость клика', 'Средняя стоимость клика, ₽'],
        'ДРР Ozon, %': ['drr', 'ДРР в продвижении', 'ДРР в продвижении, %', 'ДРР, %'],
        'ДРР общий Ozon, %': ['overallDrr', 'ДРР (общий)', 'ДРР (общий), %', 'Общий ДРР'],
        'Расход, ₽': ['expense', 'Расход', 'Расход, ₽, с НДС', 'Расход, Р, с НДС'],
        'Цена, ₽': ['price', 'Цена товара', 'Цена товара, ₽', 'Цена товара, Р']}.items():
        source = require_col(stats, aliases, target) if target in ('Продажи, ₽', 'Расход, ₽') else get_col(stats, aliases)
        data[target] = number(source)
    data = data[data['SKU'].str.strip().ne('')]
    data = data[~data['SKU'].str.casefold().isin(['всего', 'итого', 'total'])]
    grouped = data.groupby(['Кампания ID', 'SKU'], as_index=False).agg({
        'Показы': 'sum', 'Клики': 'sum', 'Корзины': 'sum', 'Заказы': 'sum',
        'Продажи, ₽': 'sum', 'Заказы модели': 'sum', 'Продажи модели, ₽': 'sum',
        'Заказано всего, ₽': 'sum', 'Расход, ₽': 'sum', 'Цена, ₽': 'max',
        'Средний CPC Ozon, ₽': 'max', 'ДРР Ozon, %': 'max', 'ДРР общий Ozon, %': 'max',
        'Дата': 'max', 'Дата добавления': 'max', 'Товар из отчёта': 'max'
    })
    current = pd.DataFrame(index=products.index)
    current['Кампания ID'] = text_id(get_col(products, ['campaignId']))
    current['SKU'] = text_id(get_col(products, ['sku']))
    current['Товар'] = get_col(products, ['title', 'Название товара'])
    current['Текущая ставка CPC, ₽'] = number(get_col(products, ['bid'])) / 1_000_000
    current['Целевой ДРР стратегии, %'] = number(get_col(products, ['targetCir']))
    current['Целевая позиция'] = get_col(products, ['topPosition'])
    grouped = grouped.merge(current.drop_duplicates(['Кампания ID', 'SKU']), on=['Кампания ID', 'SKU'], how='left')
    grouped['Товар'] = grouped['Товар'].replace('', pd.NA).fillna(grouped['Товар из отчёта'])
    grouped = grouped.drop(columns=['Товар из отчёта'])
    if not competitive.empty:
        comp = pd.DataFrame(index=competitive.index)
        comp['Кампания ID'] = text_id(get_col(competitive, ['campaignId']))
        comp['SKU'] = text_id(get_col(competitive, ['sku']))
        comp['Конкурентная ставка, ₽'] = number(get_col(competitive, ['bid'])) / 1_000_000
        grouped = grouped.merge(comp.drop_duplicates(['Кампания ID', 'SKU']), on=['Кампания ID', 'SKU'], how='left')
    names = dict(zip(text_id(get_col(campaigns, ['id'])), get_col(campaigns, ['title', 'Название'])))
    grouped.insert(1, 'Кампания', grouped['Кампания ID'].map(names).fillna(''))
    grouped['CTR %'] = [pct(c, v) for c, v in zip(grouped['Клики'], grouped['Показы'])]
    grouped['CPC, ₽'] = [round(s / c, 2) if c else 0 for s, c in zip(grouped['Расход, ₽'], grouped['Клики'])]
    grouped['CR клик→заказ %'] = [pct(o, c) for o, c in zip(grouped['Заказы'], grouped['Клики'])]
    grouped['CPO, ₽'] = [round(s / o, 2) if o else 0 for s, o in zip(grouped['Расход, ₽'], grouped['Заказы'])]
    grouped['ДРР %'] = [pct(s, r) for s, r in zip(grouped['Расход, ₽'], grouped['Продажи, ₽'])]
    grouped['ДРР модели расчётный, %'] = [pct(s, r) for s, r in zip(grouped['Расход, ₽'], grouped['Продажи модели, ₽'])]
    grouped['ДРР общий расчётный, %'] = [pct(s, r) for s, r in zip(grouped['Расход, ₽'], grouped['Заказано всего, ₽'])]
    grouped['Рекомендация'] = [verdict(s, r, o, c, TARGET_DRR) for s, r, o, c in zip(
        grouped['Расход, ₽'], grouped['Продажи, ₽'], grouped['Заказы'], grouped['Клики'])]
    return grouped.sort_values('Расход, ₽', ascending=False)

def cpo_product_table(report, current):
    source = report if not report.empty else current
    if source.empty:
        return pd.DataFrame()
    result = pd.DataFrame(index=source.index)
    result['SKU'] = text_id(get_col(source, ['sku', 'SKU']))
    result['Артикул продавца'] = get_col(source, ['sourceSku', 'offerId', 'OfferID', 'Артикул'])
    result['Товар'] = get_col(source, ['title', 'Наименование', 'Название товара'])
    result['Категория'] = get_col(source, ['category', 'Категория'])
    for target, aliases in {
        'Цена, ₽': ['price', 'Цена товара', 'Цена товара, ₽'], 'Ставка CPO, %': ['bid', 'Ставка, %'],
        'Ставка CPO, ₽': ['bidPrice', 'bidValue', 'BidValue', 'Ставка, ₽'], 'Заказы': ['orders', 'Количество заказов'],
        'Продажи, ₽': ['sales', 'ordersMoney', 'OrdersMoney', 'Сумма заказов', 'Сумма заказов, ₽', 'Стоимость заказов'],
        'Расход, ₽': ['expense', 'moneySpent', 'MoneySpent', 'Расход', 'Расход, ₽'], 'Корзины': ['toCart', 'В корзину'],
        'ДРР Ozon, %': ['drr', 'ДРР, %']}.items():
        metric = require_col(source, aliases, target) if not report.empty and target in ('Продажи, ₽', 'Расход, ₽') else get_col(source, aliases)
        result[target] = number(metric)
    if not current.empty:
        cur = pd.DataFrame(index=current.index)
        cur['SKU'] = text_id(get_col(current, ['sku']))
        cur['Товар текущий'] = get_col(current, ['title'])
        cur['Продвижение включено'] = get_col(current, ['searchPromoStatus'])
        cur['Индекс видимости'] = get_col(current, ['visibilityIndex'])
        cur['Мин. ставка CPO'] = number(get_col(current, ['minBid']))
        cur['Текущая ставка CPO, %'] = number(get_col(current, ['bid']))
        cur['Текущая ставка CPO, ₽'] = number(get_col(current, ['bidPrice']))
        result = result.merge(cur.drop_duplicates('SKU'), on='SKU', how='outer')
        result['Товар'] = result['Товар'].replace('', pd.NA).fillna(result['Товар текущий'])
        result = result.drop(columns=['Товар текущий'])
    for column in ['Заказы', 'Продажи, ₽', 'Расход, ₽', 'Корзины']:
        if column not in result:
            result[column] = 0.0
        result[column] = pd.to_numeric(result[column], errors='coerce').fillna(0)
    result['CPO, ₽'] = [round(s / o, 2) if o else 0 for s, o in zip(result['Расход, ₽'], result['Заказы'])]
    result['ДРР расчётный, %'] = [pct(s, r) for s, r in zip(result['Расход, ₽'], result['Продажи, ₽'])]
    result['Рекомендация'] = [verdict(s, r, o, MIN_CLICKS_FOR_VERDICT, TARGET_DRR) for s, r, o in zip(
        result['Расход, ₽'], result['Продажи, ₽'], result['Заказы'])]
    return result.sort_values('Расход, ₽', ascending=False)

def all_sku_cpo_table(report):
    if report.empty:
        return pd.DataFrame()
    result = pd.DataFrame(index=report.index)
    result['Дата'] = get_col(report, ['date', 'Дата'])
    result['Статус продвижения'] = get_col(report, ['status', 'Статус продвижения', 'Продвижение'])
    for target, aliases in {
        'Расход, ₽': ['expense', 'Расход', 'Расход, ₽'],
        'Продажи из поиска, ₽': ['salesFromSearch', 'Продажи из поиска', 'Продажи из поиска, ₽'],
        'Продажи из рекомендаций, ₽': ['salesFromRecommendations', 'Продажи из рекомендаций', 'Продажи из рекомендаций, ₽'],
        'Заказы из поиска': ['ordersFromSearch', 'Количество заказов из поиска', 'Заказы из поиска'],
        'Заказы из рекомендаций': ['ordersFromRecommendations', 'Количество заказов из рекомендаций', 'Заказы из рекомендаций'],
    }.items():
        result[target] = number(get_col(report, aliases))
    result['Продажи всего, ₽'] = result['Продажи из поиска, ₽'] + result['Продажи из рекомендаций, ₽']
    result['Заказы всего'] = result['Заказы из поиска'] + result['Заказы из рекомендаций']
    result['ДРР расчётный, %'] = [pct(s, r) for s, r in zip(result['Расход, ₽'], result['Продажи всего, ₽'])]
    return result

def build_summary(campaigns, cpc_result, cpo_result, all_sku_cpo_result, d_from, d_to):
    cpc = cpc_result if not cpc_result.empty else pd.DataFrame()
    cpc_spend = cpc['Расход, ₽'].sum() if 'Расход, ₽' in cpc else 0
    cpc_sales = cpc['Продажи, ₽'].sum() if 'Продажи, ₽' in cpc else 0
    cpc_orders = cpc['Заказы'].sum() if 'Заказы' in cpc else 0
    cpo_spend = cpo_result['Расход, ₽'].sum() if 'Расход, ₽' in cpo_result else 0
    cpo_sales = cpo_result['Продажи, ₽'].sum() if 'Продажи, ₽' in cpo_result else 0
    cpo_orders = cpo_result['Заказы'].sum() if 'Заказы' in cpo_result else 0
    all_cpo_spend = all_sku_cpo_result['Расход, ₽'].sum() if 'Расход, ₽' in all_sku_cpo_result else 0
    all_cpo_sales = all_sku_cpo_result['Продажи всего, ₽'].sum() if 'Продажи всего, ₽' in all_sku_cpo_result else 0
    all_cpo_orders = all_sku_cpo_result['Заказы всего'].sum() if 'Заказы всего' in all_sku_cpo_result else 0
    total_spend = cpc_spend + cpo_spend + all_cpo_spend
    total_sales = cpc_sales + cpo_sales + all_cpo_sales
    return pd.DataFrame([
        ('Период', f'{d_from} — {d_to}'), ('Целевой ДРР, %', TARGET_DRR),
        ('Кампаний всего', len(campaigns)), ('Расход CPC, ₽', round(cpc_spend, 2)),
        ('Продажи CPC, ₽', round(cpc_sales, 2)), ('Заказы CPC', int(cpc_orders)),
        ('ДРР CPC, %', pct(cpc_spend, cpc_sales)), ('Расход CPO — выбранные товары, ₽', round(cpo_spend, 2)),
        ('Продажи CPO — выбранные товары, ₽', round(cpo_sales, 2)), ('Заказы CPO — выбранные товары', int(cpo_orders)),
        ('ДРР CPO — выбранные товары, %', pct(cpo_spend, cpo_sales)),
        ('Расход CPO — все товары Plus, ₽', round(all_cpo_spend, 2)),
        ('Продажи CPO — все товары Plus, ₽', round(all_cpo_sales, 2)),
        ('Заказы CPO — все товары Plus', int(all_cpo_orders)),
        ('ДРР CPO — все товары Plus, %', pct(all_cpo_spend, all_cpo_sales)),
        ('Расход всего, ₽', round(total_spend, 2)),
        ('Продажи всего, ₽', round(total_sales, 2)),
        ('ДРР общий, %', pct(total_spend, total_sales)),
    ], columns=['Показатель', 'Значение'])

OZON_STATUS_NAMES = {
    'CAMPAIGN_STATE_RUNNING': 'активна', 'CAMPAIGN_STATE_INACTIVE': 'неактивна',
    'CAMPAIGN_STATE_STOPPED': 'остановлена', 'CAMPAIGN_STATE_PLANNED': 'запланирована',
    'CAMPAIGN_STATE_ENDED': 'завершена', 'CAMPAIGN_STATE_ARCHIVED': 'в архиве',
}
OZON_PAYMENT_NAMES = {'CPC': 'за клик', 'CPM': 'за показы', 'CPO': 'за заказ'}
OZON_STRATEGY_NAMES = {
    'MAX_CLICKS': 'максимум кликов', 'TOP_MAX_CLICKS': 'максимум кликов в поиске',
    'TARGET_BIDS': 'средняя цена клика', 'TOP_PROMOTION': 'вывод в топ',
    'TARGET_CIR': 'целевой ДРР', 'NO_AUTO_STRATEGY': 'ручная',
}

def compact_campaign_table(source):
    columns = ['ID', 'Название', 'Статус', 'Оплата', 'Стратегия', 'Показы', 'Клики', 'CTR %',
               'CPC', 'Корзины', 'Заказы', 'CR клик->заказ %', 'Выручка', 'Затраты',
               'ДРР %', 'CPO', 'Вердикт']
    if source.empty:
        return pd.DataFrame(columns=columns)
    metric_columns = ['Показы', 'Клики', 'Корзины', 'Заказы', 'Продажи, ₽', 'Расход, ₽']
    active = source[metric_columns].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).gt(0)
    data = source.loc[active].copy()
    result = pd.DataFrame(index=data.index)
    result['ID'] = text_id(data['ID'])
    result['Название'] = data['Название']
    result['Статус'] = data['Статус'].map(OZON_STATUS_NAMES).fillna(data['Статус'])
    result['Оплата'] = data['Модель оплаты'].astype(str).str.upper().map(OZON_PAYMENT_NAMES).fillna(data['Модель оплаты'])
    result['Стратегия'] = data['Стратегия'].map(OZON_STRATEGY_NAMES).fillna(data['Стратегия'])
    for column in ['Показы', 'Клики', 'Корзины', 'Заказы']:
        result[column] = pd.to_numeric(data[column], errors='coerce').fillna(0).astype(int)
    result['CTR %'] = data['CTR %']
    result['CPC'] = data['CPC, ₽']
    result['CR клик->заказ %'] = data['CR клик→заказ %']
    result['Выручка'] = data['Продажи, ₽'].round(1)
    result['Затраты'] = data['Расход, ₽'].round(1)
    result['ДРР %'] = data['ДРР %']
    result['CPO'] = data['CPO, ₽']
    result['Вердикт'] = data['Вердикт'].str.replace(' / масштабировать', '', regex=False).str.replace(' / снизить расход', '', regex=False)
    return result[columns].sort_values('Затраты', ascending=False)

def bid_label(value, unit):
    value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(value) or value <= 0:
        return ''
    return f'{value:g} {unit}'

def compact_product_table(cpc, cpo):
    columns = ['Кампания ID', 'Кампания', 'SKU', 'Товар', 'Тип рекламы', 'Показы', 'Клики',
               'CTR %', 'Корзины', 'Заказы', 'Выручка', 'Затраты', 'CPC', 'CPO',
               'ДРР %', 'Ставка', 'Рекомендация']
    parts = []
    if not cpc.empty:
        part = pd.DataFrame(index=cpc.index)
        part['Кампания ID'] = text_id(cpc['Кампания ID'])
        part['Кампания'] = cpc['Кампания']
        part['SKU'] = text_id(cpc['SKU'])
        part['Товар'] = cpc['Товар']
        part['Тип рекламы'] = 'Оплата за клик'
        for column in ['Показы', 'Клики', 'Корзины', 'Заказы']:
            part[column] = pd.to_numeric(cpc[column], errors='coerce').fillna(0).astype(int)
        part['CTR %'] = cpc['CTR %']
        part['Выручка'] = cpc['Продажи, ₽'].round(1)
        part['Затраты'] = cpc['Расход, ₽'].round(1)
        part['CPC'] = cpc['CPC, ₽']
        part['CPO'] = cpc['CPO, ₽']
        part['ДРР %'] = cpc['ДРР %']
        part['Ставка'] = [bid_label(value, '₽') for value in cpc.get('Текущая ставка CPC, ₽', pd.Series(0, index=cpc.index))]
        part['Рекомендация'] = cpc['Рекомендация']
        parts.append(part)
    if not cpo.empty:
        part = pd.DataFrame(index=cpo.index)
        part['Кампания ID'] = ''
        part['Кампания'] = 'Оплата за заказ'
        part['SKU'] = text_id(cpo['SKU'])
        part['Товар'] = cpo['Товар']
        part['Тип рекламы'] = 'Оплата за заказ'
        part['Показы'] = 0
        part['Клики'] = 0
        part['CTR %'] = 0.0
        part['Корзины'] = pd.to_numeric(cpo['Корзины'], errors='coerce').fillna(0).astype(int)
        part['Заказы'] = pd.to_numeric(cpo['Заказы'], errors='coerce').fillna(0).astype(int)
        part['Выручка'] = cpo['Продажи, ₽'].round(1)
        part['Затраты'] = cpo['Расход, ₽'].round(1)
        part['CPC'] = 0.0
        part['CPO'] = cpo['CPO, ₽']
        part['ДРР %'] = cpo['ДРР расчётный, %']
        bids = cpo.get('Текущая ставка CPO, %', cpo.get('Ставка CPO, %', pd.Series(0, index=cpo.index)))
        part['Ставка'] = [bid_label(value, '%') for value in bids]
        part['Рекомендация'] = cpo['Рекомендация']
        parts.append(part)
    if not parts:
        return pd.DataFrame(columns=columns)
    result = pd.concat(parts, ignore_index=True)
    metrics = result[['Показы', 'Клики', 'Корзины', 'Заказы', 'Выручка', 'Затраты']].fillna(0).sum(axis=1)
    result = result[metrics.gt(0)]
    return result[columns].sort_values('Затраты', ascending=False)

def compact_phrase_table(source, campaigns):
    columns = ['Кампания ID', 'Кампания', 'SKU', 'Поисковый запрос', 'Показы', 'Клики', 'CTR %',
               'Корзины', 'Заказы', 'Выручка', 'Затраты', 'CPC', 'CPO', 'ДРР %', 'Рекомендация']
    if source.empty:
        return pd.DataFrame(columns=columns)
    result = pd.DataFrame(index=source.index)
    result['Кампания ID'] = text_id(get_col(source, ['campaignId', 'ID кампании', 'Кампания ID']))
    names = dict(zip(text_id(get_col(campaigns, ['id'])), get_col(campaigns, ['title', 'Название'])))
    result['Кампания'] = result['Кампания ID'].map(names).fillna(get_col(source, ['campaignTitle', 'Название кампании']))
    result['SKU'] = text_id(get_col(source, ['sku', 'SKU']))
    result['Поисковый запрос'] = get_col(source, ['phrase', 'query', 'searchPhrase', 'Поисковый запрос', 'Поисковая фраза', 'Ключевая фраза', 'Запрос', 'Фраза'])
    for target, aliases in {
        'Показы': ['views', 'Показы'], 'Клики': ['clicks', 'Клики'],
        'Корзины': ['toCart', 'В корзину', 'Добавления в корзину'],
        'Заказы': ['orders', 'Заказы', 'Продано товаров'],
        'Выручка': ['sales', 'Продажи', 'Продажи в продвижении', 'Продажи в продвижении, ₽'],
        'Затраты': ['expense', 'spend', 'Расход', 'Расход, ₽', 'Расход, ₽, с НДС', 'Затраты'],
    }.items():
        result[target] = number(get_col(source, aliases))
    result = result[result['Поисковый запрос'].astype(str).str.strip().ne('')]
    if result.empty:
        return pd.DataFrame(columns=columns)
    result['CTR %'] = [pct(c, v) for c, v in zip(result['Клики'], result['Показы'])]
    result['CPC'] = [round(s / c, 1) if c else 0 for s, c in zip(result['Затраты'], result['Клики'])]
    result['CPO'] = [round(s / o) if o else '' for s, o in zip(result['Затраты'], result['Заказы'])]
    result['ДРР %'] = [pct(s, r) if r else '' for s, r in zip(result['Затраты'], result['Выручка'])]
    result['Рекомендация'] = [verdict(s, r, o, c, TARGET_DRR) for s, r, o, c in zip(
        result['Затраты'], result['Выручка'], result['Заказы'], result['Клики'])]
    return result[columns].sort_values('Затраты', ascending=False)

def compact_summary(campaigns, products, all_sku_cpo, d_from, d_to):
    spend = products['Затраты'].sum() if 'Затраты' in products else 0
    sales = products['Выручка'].sum() if 'Выручка' in products else 0
    orders = products['Заказы'].sum() if 'Заказы' in products else 0
    if not all_sku_cpo.empty:
        spend += all_sku_cpo['Расход, ₽'].sum()
        sales += all_sku_cpo['Продажи всего, ₽'].sum()
        orders += all_sku_cpo['Заказы всего'].sum()
    verdicts = campaigns['Вердикт'].value_counts().to_dict() if not campaigns.empty else {}
    rows = [
        ('Период', f'{d_from} - {d_to}'), ('Целевой ДРР, %', TARGET_DRR),
        ('Кампаний со статистикой', len(campaigns)), ('Товаров со статистикой', len(products)),
        ('Затраты на рекламу, всего', round(spend)),
        ('Выручка с рекламы (заказы), всего', round(sales)),
        ('ДРР общий, %', pct(spend, sales)), ('Заказов с рекламы, шт', int(orders)),
    ]
    rows.extend((f'Кампаний: {name}', count) for name, count in verdicts.items())
    return pd.DataFrame(rows, columns=['Показатель', 'Значение'])

def write_excel(path, sheets):
    from openpyxl.styles import Font, PatternFill
    from openpyxl.utils import get_column_letter
    def excel_value(value):
        if isinstance(value, (list, dict, tuple, set)):
            return json.dumps(value, ensure_ascii=False, default=str)
        return value
    written = {}
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        for name, frame in sheets.items():
            output = (frame if frame is not None else pd.DataFrame()).copy()
            for column in output.select_dtypes(include='object').columns:
                output[column] = output[column].map(excel_value)
            try:
                output.to_excel(writer, sheet_name=name[:31], index=False)
            except Exception as exc:
                print(f'  Внимание — лист {name} не экспортирован полностью: {exc}')
                output = pd.DataFrame({'Ошибка экспорта листа': [str(exc)]})
                output.to_excel(writer, sheet_name=name[:31], index=False)
            written[name] = output
        good = PatternFill('solid', start_color='C6EFCE')
        middle = PatternFill('solid', start_color='FFEB9C')
        bad = PatternFill('solid', start_color='FFC7CE')
        for name, frame in written.items():
            ws = writer.book[name[:31]]
            ws.freeze_panes = 'A2'
            if ws.max_column:
                for cell in ws[1]:
                    cell.font = Font(bold=True)
            if frame is not None and not frame.empty:
                ws.auto_filter.ref = ws.dimensions
                for verdict_col in ('Вердикт', 'Рекомендация', 'Экономика', 'Вердикт экономики'):
                    if verdict_col not in frame.columns:
                        continue
                    index = list(frame.columns).index(verdict_col) + 1
                    for row in range(2, ws.max_row + 1):
                        cell = ws.cell(row=row, column=index)
                        value = str(cell.value or '')
                        if value.startswith(('Эффективна', 'Масштабировать', 'В плюс')):
                            cell.fill = good
                        elif value.startswith(('Наблюдать', 'Мало данных', 'Нет затрат', 'Около нуля', 'Нет данных', 'Нет заказов')):
                            cell.fill = middle
                        elif value.startswith(('Неэффективна', 'Отключить', 'В минус')):
                            cell.fill = bad
            for index, column in enumerate(ws.columns, 1):
                width = max((len(str(cell.value)) for cell in list(column)[:200] if cell.value is not None), default=8)
                ws.column_dimensions[get_column_letter(index)].width = min(width + 2, 45)
    print('Готово:', path)

# ---------------------------------------------------------------------------
# Юнит-экономика и предельный ДРР
# ---------------------------------------------------------------------------

def read_cost_table(path):
    """Файл себестоимости: колонки «артикул»/«sku» и «себестоимость» (разделитель ; или ,)."""
    from pathlib import Path
    if not path or not Path(path).exists():
        return pd.DataFrame(columns=['offer_id', 'sku', 'Себестоимость файла, ₽'])
    raw = Path(path).read_bytes()
    text = decode_ozon_text(raw)
    separator = ';' if text.count(';') >= text.count(',') else ','
    frame = pd.read_csv(io.StringIO(text), sep=separator, dtype=str)
    frame.columns = [str(column).strip() for column in frame.columns]
    key_offer = get_col(frame, ['offer_id', 'артикул', 'артикул продавца', 'offer'])
    key_sku = get_col(frame, ['sku', 'ozon sku', 'озон sku', 'sku ozon'])
    cost = get_col(frame, ['себестоимость', 'себестоимость, ₽', 'cost', 'cost_price', 'закупка', 'закупочная цена'])
    result = pd.DataFrame({
        'offer_id': key_offer.astype(str).str.strip() if not key_offer.empty else '',
        'sku': text_id(key_sku) if not key_sku.empty else '',
        'Себестоимость файла, ₽': number(cost) if not cost.empty else pd.NA,
    })
    result = result[pd.to_numeric(result['Себестоимость файла, ₽'], errors='coerce').gt(0)]
    print(f'Файл себестоимости «{path}»: {len(result)} строк.')
    return result

def build_ad_sku(cpc_full, cpo_full):
    """Итог рекламы по каждому SKU: заказы, выручка и расход по всем кампаниям вместе."""
    parts = []
    if not cpc_full.empty:
        part = cpc_full[['SKU', 'Заказы', 'Продажи, ₽', 'Расход, ₽']].copy()
        part['Товар'] = cpc_full.get('Товар', '')
        parts.append(part)
    if not cpo_full.empty:
        part = cpo_full[['SKU', 'Заказы', 'Продажи, ₽', 'Расход, ₽']].copy()
        part['Товар'] = cpo_full.get('Товар', '')
        parts.append(part)
    if not parts:
        return pd.DataFrame(columns=['SKU', 'Товар', 'Заказы', 'Продажи, ₽', 'Расход, ₽'])
    data = pd.concat(parts, ignore_index=True)
    data['SKU'] = text_id(data['SKU'])
    data = data[data['SKU'].ne('')]
    for column in ['Заказы', 'Продажи, ₽', 'Расход, ₽']:
        data[column] = pd.to_numeric(data[column], errors='coerce').fillna(0)
    grouped = data.groupby('SKU', as_index=False).agg(
        {'Заказы': 'sum', 'Продажи, ₽': 'sum', 'Расход, ₽': 'sum', 'Товар': 'max'})
    return grouped[grouped[['Заказы', 'Продажи, ₽', 'Расход, ₽']].sum(axis=1).gt(0)]

def scheme_costs(prices, scheme):
    """Комиссия %, логистика и эквайринг из v5/product/info/prices для схемы FBO или FBS."""
    tag = scheme.lower()
    commission_pct = number(get_col(prices, [f'commissions.sales_percent_{tag}']))
    trans_min = number(get_col(prices, [f'commissions.{tag}_direct_flow_trans_min_amount']))
    trans_max = number(get_col(prices, [f'commissions.{tag}_direct_flow_trans_max_amount']))
    last_mile = number(get_col(prices, [f'commissions.{tag}_deliv_to_customer_amount']))
    logistics = (trans_min + trans_max) / 2 + last_mile
    if tag == 'fbs':
        first_min = number(get_col(prices, ['commissions.fbs_first_mile_min_amount']))
        first_max = number(get_col(prices, ['commissions.fbs_first_mile_max_amount']))
        logistics = logistics + (first_min + first_max) / 2
    acquiring = number(get_col(prices, ['acquiring']))
    return commission_pct, logistics, acquiring

def econ_verdict(fact_drr, breakeven_drr, orders):
    if pd.isna(breakeven_drr):
        return 'Нет данных'
    if not orders:
        return 'Нет заказов'
    if fact_drr <= breakeven_drr - BREAKEVEN_SAFETY_PP:
        return 'В плюс'
    if fact_drr <= breakeven_drr + BREAKEVEN_SAFETY_PP:
        return 'Около нуля'
    return 'В минус'

def economics_table(prices, product_index, ad_sku, cost_table):
    """Юнит-экономика по каждому SKU из рекламы: маржа до рекламы и предельный ДРР."""
    if ad_sku.empty:
        return pd.DataFrame()
    result = ad_sku.rename(columns={
        'Заказы': 'Реклама: заказы', 'Продажи, ₽': 'Реклама: выручка, ₽', 'Расход, ₽': 'Реклама: расход, ₽'}).copy()

    card = pd.DataFrame()
    if not prices.empty:
        card = pd.DataFrame({
            'product_id': text_id(get_col(prices, ['product_id'])),
            'offer_id': get_col(prices, ['offer_id']).astype(str).str.strip(),
            'Цена карточки, ₽': number(get_col(prices, ['price.price', 'price.marketing_seller_price'])),
        })
        commission_pct, logistics, acquiring = scheme_costs(prices, SALES_SCHEME)
        card['Комиссия Ozon, %'] = commission_pct
        card['Логистика, ₽'] = logistics.round(2)
        card['Эквайринг, ₽'] = acquiring.round(2)
        if not product_index.empty:
            index = product_index.copy()
            index['product_id'] = text_id(index['product_id'])
            index['sku'] = text_id(index['sku'])
            card = card.merge(index[['product_id', 'sku', 'offer_id', 'name']].drop_duplicates('product_id'),
                              on='product_id', how='left', suffixes=('', '_idx'))
            card['offer_id'] = card['offer_id'].replace('', pd.NA).fillna(card.get('offer_id_idx'))
        card = card.rename(columns={'sku': 'SKU', 'offer_id': 'Артикул', 'name': 'Название карточки'})
        card['SKU'] = text_id(card.get('SKU', pd.Series('', index=card.index)))
        card = card[card['SKU'].ne('')].drop_duplicates('SKU')
        keep = ['SKU', 'Артикул', 'Название карточки', 'Цена карточки, ₽', 'Комиссия Ozon, %', 'Логистика, ₽', 'Эквайринг, ₽']
        result = result.merge(card[[c for c in keep if c in card.columns]], on='SKU', how='left')

    for column, default in [('Артикул', ''), ('Название карточки', ''), ('Цена карточки, ₽', pd.NA),
                            ('Комиссия Ozon, %', pd.NA), ('Логистика, ₽', pd.NA), ('Эквайринг, ₽', pd.NA)]:
        if column not in result:
            result[column] = default
    result['Товар'] = result['Товар'].replace('', pd.NA).fillna(result['Название карточки']).fillna('')
    result = result.drop(columns=['Название карточки'])

    orders = pd.to_numeric(result['Реклама: заказы'], errors='coerce').fillna(0)
    ad_sales = pd.to_numeric(result['Реклама: выручка, ₽'], errors='coerce').fillna(0)
    card_price = pd.to_numeric(result['Цена карточки, ₽'], errors='coerce')
    avg_price = pd.Series(pd.NA, index=result.index, dtype='object')
    has_orders = orders.gt(0) & ad_sales.gt(0)
    avg_price[has_orders] = (ad_sales[has_orders] / orders[has_orders]).round(2)
    avg_price[~has_orders] = card_price[~has_orders]
    result['Средняя цена продажи, ₽'] = pd.to_numeric(avg_price, errors='coerce')

    price = result['Средняя цена продажи, ₽']
    commission_rub = (price * pd.to_numeric(result['Комиссия Ozon, %'], errors='coerce') / 100).round(2)
    result['Комиссия Ozon, ₽'] = commission_rub
    result['Налог, ₽'] = (price * TAX_RATE_PCT / 100).round(2)

    costs = pd.Series(pd.NA, index=result.index, dtype='object')
    source = pd.Series('нет данных', index=result.index)
    if not cost_table.empty:
        by_sku = cost_table[cost_table['sku'].astype(str).ne('')].drop_duplicates('sku').set_index('sku')['Себестоимость файла, ₽']
        by_offer = cost_table[cost_table['offer_id'].astype(str).ne('')].drop_duplicates('offer_id').set_index('offer_id')['Себестоимость файла, ₽']
        from_sku = result['SKU'].map(by_sku)
        from_offer = result['Артикул'].astype(str).map(by_offer)
        file_cost = pd.to_numeric(from_sku.fillna(from_offer), errors='coerce')
        costs[file_cost.notna()] = file_cost[file_cost.notna()]
        source[file_cost.notna()] = 'файл'
    if DEFAULT_COST_SHARE_PCT > 0:
        assumed = (price * DEFAULT_COST_SHARE_PCT / 100).round(2)
        need = pd.to_numeric(costs, errors='coerce').isna() & assumed.notna()
        costs[need] = assumed[need]
        source[need] = f'{DEFAULT_COST_SHARE_PCT:g}% цены (допущение)'
    result['Себестоимость, ₽'] = pd.to_numeric(costs, errors='coerce')
    result['Источник себестоимости'] = source
    result['Доп. затраты, ₽'] = EXTRA_COST_PER_UNIT

    margin = (price - commission_rub - pd.to_numeric(result['Логистика, ₽'], errors='coerce')
              - pd.to_numeric(result['Эквайринг, ₽'], errors='coerce') - result['Налог, ₽']
              - result['Себестоимость, ₽'] - EXTRA_COST_PER_UNIT)
    result['Маржа до рекламы, ₽'] = margin.round(2)
    result['Маржа до рекламы, %'] = (margin / price * 100).round(1)
    result['Предельный ДРР, %'] = result['Маржа до рекламы, %']
    result['Предельный CPO, ₽'] = result['Маржа до рекламы, ₽']

    spend = pd.to_numeric(result['Реклама: расход, ₽'], errors='coerce').fillna(0)
    result['Факт ДРР, %'] = [pct(s, r) if r else pd.NA for s, r in zip(spend, ad_sales)]
    result['Факт CPO, ₽'] = [round(s / o, 2) if o else pd.NA for s, o in zip(spend, orders)]
    result['Запас ДРР, п.п.'] = (pd.to_numeric(result['Предельный ДРР, %'], errors='coerce')
                                 - pd.to_numeric(result['Факт ДРР, %'], errors='coerce')).round(1)
    result['Прибыль с рекламы, ₽'] = (margin * orders - spend).round(2)
    result['Вердикт экономики'] = [
        econ_verdict(f if pd.notna(f) else 0.0, b, o)
        for f, b, o in zip(pd.to_numeric(result['Факт ДРР, %'], errors='coerce'),
                           pd.to_numeric(result['Предельный ДРР, %'], errors='coerce'), orders)]
    columns = ['SKU', 'Артикул', 'Товар', 'Средняя цена продажи, ₽', 'Цена карточки, ₽',
               'Комиссия Ozon, %', 'Комиссия Ozon, ₽', 'Логистика, ₽', 'Эквайринг, ₽', 'Налог, ₽',
               'Себестоимость, ₽', 'Источник себестоимости', 'Доп. затраты, ₽',
               'Маржа до рекламы, ₽', 'Маржа до рекламы, %', 'Предельный ДРР, %', 'Предельный CPO, ₽',
               'Реклама: расход, ₽', 'Реклама: заказы', 'Реклама: выручка, ₽',
               'Факт ДРР, %', 'Факт CPO, ₽', 'Запас ДРР, п.п.', 'Прибыль с рекламы, ₽', 'Вердикт экономики']
    return result[columns].sort_values('Реклама: расход, ₽', ascending=False)

def apply_campaign_economics(campaign_result, cpc_full, econ):
    """Добавляет в лист «Кампании» предельный ДРР, запас и прибыль, взвешенные по SKU кампании."""
    if campaign_result.empty:
        return campaign_result
    result = campaign_result.copy()
    for column in ['Предельный ДРР %', 'Запас ДРР, п.п.', 'Прибыль с рекламы', 'Покрытие экономикой %']:
        result[column] = pd.NA
    result['Экономика'] = 'Нет данных'
    if econ.empty or cpc_full.empty:
        return result
    margins = econ.set_index('SKU')['Маржа до рекламы, ₽']
    rows = cpc_full[['Кампания ID', 'SKU', 'Заказы', 'Продажи, ₽', 'Расход, ₽']].copy()
    rows['Кампания ID'] = text_id(rows['Кампания ID'])
    rows['SKU'] = text_id(rows['SKU'])
    for column in ['Заказы', 'Продажи, ₽', 'Расход, ₽']:
        rows[column] = pd.to_numeric(rows[column], errors='coerce').fillna(0)
    rows['Маржа, ₽'] = pd.to_numeric(rows['SKU'].map(margins), errors='coerce')
    for campaign_id, group in rows.groupby('Кампания ID'):
        mask = result['ID'].astype(str).eq(campaign_id)
        if not mask.any():
            continue
        known = group[group['Маржа, ₽'].notna()]
        sales_total = group['Продажи, ₽'].sum()
        sales_known = known['Продажи, ₽'].sum()
        coverage = pct(sales_known, sales_total) if sales_total else 0.0
        result.loc[mask, 'Покрытие экономикой %'] = coverage
        if known.empty or sales_known <= 0:
            continue
        margin_sum = (known['Маржа, ₽'] * known['Заказы']).sum()
        spend_known = known['Расход, ₽'].sum()
        breakeven = margin_sum / sales_known * 100
        fact = spend_known / sales_known * 100
        result.loc[mask, 'Предельный ДРР %'] = round(breakeven, 1)
        result.loc[mask, 'Запас ДРР, п.п.'] = round(breakeven - fact, 1)
        result.loc[mask, 'Прибыль с рекламы'] = round(margin_sum - spend_known, 1)
        result.loc[mask, 'Экономика'] = econ_verdict(fact, breakeven, known['Заказы'].sum())
    return result

FUNNEL_COLUMN_NAMES = {
    'session_view_pdp': 'Переходы в карточку', 'hits_view_pdp': 'Показы карточки',
    'hits_tocart': 'Корзины', 'ordered_units': 'Заказано, шт', 'revenue': 'Заказано, ₽',
    'delivered_units': 'Доставлено, шт', 'returns': 'Возвраты, шт', 'cancellations': 'Отмены, шт',
    'position_category': 'Позиция в поиске', 'conv_tocart_pdp': 'CR карточка→корзина % (Ozon)',
}

def funnel_sheet(funnel_raw, metrics_used, product_index, ad_sku, econ):
    """Лист «Воронка»: продажи всего магазина по SKU + вклад рекламы, как в отчёте WB."""
    if funnel_raw.empty:
        return pd.DataFrame()
    result = pd.DataFrame()
    result['SKU'] = text_id(funnel_raw['sku'])
    result['Товар'] = funnel_raw.get('name', '')
    if not product_index.empty:
        index = product_index.copy()
        index['sku'] = text_id(index['sku'])
        offers = index.drop_duplicates('sku').set_index('sku')['offer_id']
        result.insert(1, 'Артикул', result['SKU'].map(offers).fillna(''))
    for metric in metrics_used:
        title = FUNNEL_COLUMN_NAMES.get(metric, metric)
        result[title] = pd.to_numeric(funnel_raw.get(metric), errors='coerce').fillna(0).round(2)
    views = result.get('Переходы в карточку')
    carts = result.get('Корзины')
    orders = result.get('Заказано, шт')
    delivered = result.get('Доставлено, шт')
    if views is not None and carts is not None:
        result['CR карточка→корзина %'] = [pct(c, v) for c, v in zip(carts, views)]
    if carts is not None and orders is not None:
        result['CR корзина→заказ %'] = [pct(o, c) for o, c in zip(orders, carts)]
    if delivered is not None and orders is not None:
        result['Выкуп %'] = [pct(d, o) for d, o in zip(delivered, orders)]
    if not ad_sku.empty:
        ads = ad_sku.set_index('SKU')
        result['Реклама: затраты'] = pd.to_numeric(result['SKU'].map(ads['Расход, ₽']), errors='coerce').fillna(0).round(1)
        result['Реклама: заказы'] = pd.to_numeric(result['SKU'].map(ads['Заказы']), errors='coerce').fillna(0).astype(int)
        result['Реклама: выручка'] = pd.to_numeric(result['SKU'].map(ads['Продажи, ₽']), errors='coerce').fillna(0).round(1)
        revenue_total = pd.to_numeric(result.get('Заказано, ₽'), errors='coerce').fillna(0)
        result['ДРР по SKU %'] = [pct(s, r) for s, r in zip(result['Реклама: затраты'], revenue_total)]
        orders_total = pd.to_numeric(result.get('Заказано, шт'), errors='coerce').fillna(0)
        result['Доля рекламы в заказах %'] = [pct(a, t) for a, t in zip(result['Реклама: заказы'], orders_total)]
    if not econ.empty:
        limits = econ.set_index('SKU')['Предельный ДРР, %']
        result['Предельный ДРР %'] = pd.to_numeric(result['SKU'].map(limits), errors='coerce')
    sort_by = 'Заказано, ₽' if 'Заказано, ₽' in result.columns else result.columns[-1]
    return result.sort_values(sort_by, ascending=False)

def economics_summary_rows(econ, campaign_result):
    """Строки блока экономики для листа «Сводка»."""
    rows = []
    if not econ.empty:
        known = econ[pd.to_numeric(econ['Маржа до рекламы, ₽'], errors='coerce').notna()]
        spend = pd.to_numeric(known['Реклама: расход, ₽'], errors='coerce').fillna(0).sum()
        sales = pd.to_numeric(known['Реклама: выручка, ₽'], errors='coerce').fillna(0).sum()
        margin_sum = (pd.to_numeric(known['Маржа до рекламы, ₽'], errors='coerce').fillna(0)
                      * pd.to_numeric(known['Реклама: заказы'], errors='coerce').fillna(0)).sum()
        rows += [
            ('— Экономика —', ''),
            ('SKU в рекламе с известной экономикой', f'{len(known)} из {len(econ)}'),
            ('Предельный ДРР (средневзвеш.), %', pct(margin_sum, sales)),
            ('Фактический ДРР по этим SKU, %', pct(spend, sales)),
            ('Прибыль с рекламы (до рекламы маржа − расход), ₽', round(margin_sum - spend)),
        ]
        file_costed = known[known['Источник себестоимости'].eq('файл')]
        if len(file_costed) < len(known):
            rows.append(('SKU с себестоимостью из файла', f'{len(file_costed)} из {len(known)} (остальные — допущение)'))
    if not campaign_result.empty and 'Экономика' in campaign_result.columns:
        for name, count in campaign_result['Экономика'].value_counts().items():
            rows.append((f'Кампаний (экономика): {name}', count))
    return rows

print('Расчёты и экспорт готовы.')

In [ ]:
#@title Запуск выгрузки и сборка отчёта
d_from = dt.date.fromisoformat(DATE_FROM)
d_to = dt.date.fromisoformat(DATE_TO)
out = f'ozon_ads_report_{DATE_FROM}_{DATE_TO}.xlsx'
if d_from > d_to:
    raise ValueError('Дата начала позже даты окончания.')
if (d_to - d_from).days + 1 > 62:
    raise ValueError('Performance API разрешает не более 62 дней в одной статистической выгрузке.')
if d_to >= dt.date.today():
    print('Предупреждение: статистика за сегодня может быть неполной.')

api = OzonPerformance(OZON_PERF_CLIENT_ID, OZON_PERF_CLIENT_SECRET)
seller = OzonSeller(OZON_SELLER_CLIENT_ID, OZON_SELLER_API_KEY) if SELLER_API_AVAILABLE else None
errors = []
SCHEMA_WARNINGS.clear()

def safe(label, function, empty=None):
    try:
        return function()
    except Exception as exc:
        print(f'  Внимание — {label}: {exc}')
        errors.append({'Раздел': label, 'Ошибка': str(exc)})
        return pd.DataFrame() if empty is None else empty

campaigns = safe('Кампании', lambda: load_campaigns(api))
cpc_ids = campaign_ids_by_type(campaigns, 'SKU')
print('CPC-кампаний в кабинете:', len(cpc_ids))

# Одна историческая выгрузка используется и для кампаний, и для товаров.
cpc_sku_stats = safe('Статистика CPC', lambda: load_cpc_sku_stats(api, cpc_ids, DATE_FROM, DATE_TO))
if not cpc_sku_stats.empty:
    require_col(cpc_sku_stats, ['sales', 'Продажи в продвижении', 'Продажи в продвижении, ₽', 'Продажи'], 'Продажи CPC')
    require_col(cpc_sku_stats, ['expense', 'Расход, ₽, с НДС', 'Расход'], 'Расход CPC')
stats_ids = text_id(get_col(cpc_sku_stats, ['campaignId', 'ID кампании'])).drop_duplicates()
stats_ids = stats_ids[stats_ids.ne('')].tolist()
if INCLUDE_CURRENT_BIDS and stats_ids:
    cpc_products, _ = safe(
        'Текущие ставки CPC', lambda: load_cpc_products(api, stats_ids, competitive=False),
        empty=(pd.DataFrame(), pd.DataFrame()),
    )
else:
    cpc_products = pd.DataFrame()

if INCLUDE_CPO:
    cpo_products = safe('Товары CPO', lambda: load_cpo_products(api))
    cpo_stats = safe('Статистика CPO', lambda: load_cpo_statistics(api, DATE_FROM, DATE_TO))
    all_sku_cpo_stats = safe('CPO все товары Plus', lambda: load_all_sku_cpo_statistics(api, DATE_FROM, DATE_TO)) if INCLUDE_ALL_SKU_CPO else pd.DataFrame()
else:
    cpo_products = cpo_stats = all_sku_cpo_stats = pd.DataFrame()

phrases = safe('Поисковые фразы', lambda: load_phrases(api, campaigns, DATE_FROM, DATE_TO)) if INCLUDE_PHRASES else pd.DataFrame()

# Seller API: каталог, цены и воронка.
product_index = prices = funnel_raw = pd.DataFrame()
funnel_metrics = []
if seller is not None and (INCLUDE_ECONOMICS or INCLUDE_FUNNEL):
    product_index = safe('Каталог Seller API', lambda: load_seller_products(seller))
if seller is not None and INCLUDE_ECONOMICS:
    prices = safe('Цены и комиссии Seller API', lambda: load_seller_prices(seller))
if seller is not None and INCLUDE_FUNNEL:
    funnel_raw, funnel_metrics = safe(
        'Воронка Seller API', lambda: load_funnel(seller, DATE_FROM, DATE_TO),
        empty=(pd.DataFrame(), []),
    )

campaign_full = safe('Обработка кампаний', lambda: campaign_table(campaigns, cpc_sku_stats))
cpc_full = safe('Обработка товаров CPC', lambda: cpc_product_table(cpc_sku_stats, cpc_products, pd.DataFrame(), campaigns))
cpo_full = safe('Обработка товаров CPO', lambda: cpo_product_table(cpo_stats, cpo_products))
all_sku_cpo = safe('Обработка CPO все товары Plus', lambda: all_sku_cpo_table(all_sku_cpo_stats))

campaign_result = safe('Компактные кампании', lambda: compact_campaign_table(campaign_full))
product_result = safe('Компактные товары', lambda: compact_product_table(cpc_full, cpo_full))
phrase_result = safe('Компактные ключи', lambda: compact_phrase_table(phrases, campaigns))

# Экономика: себестоимость + цены Seller API -> маржа до рекламы и предельный ДРР.
cost_table = safe('Файл себестоимости', lambda: read_cost_table(COST_FILE)) if INCLUDE_ECONOMICS else pd.DataFrame()
ad_sku = safe('Итог рекламы по SKU', lambda: build_ad_sku(cpc_full, cpo_full))
econ = pd.DataFrame()
if INCLUDE_ECONOMICS and not prices.empty:
    econ = safe('Юнит-экономика', lambda: economics_table(prices, product_index, ad_sku, cost_table))
campaign_result = safe('Экономика кампаний', lambda: apply_campaign_economics(campaign_result, cpc_full, econ))
funnel_result = pd.DataFrame()
if not funnel_raw.empty:
    funnel_result = safe('Лист воронки', lambda: funnel_sheet(funnel_raw, funnel_metrics, product_index, ad_sku, econ))

summary = safe('Сборка сводки', lambda: compact_summary(campaign_result, product_result, all_sku_cpo, DATE_FROM, DATE_TO))
econ_rows = safe('Сводка экономики', lambda: economics_summary_rows(econ, campaign_result), empty=[])
if econ_rows:
    summary = pd.concat([summary, pd.DataFrame(econ_rows, columns=['Показатель', 'Значение'])], ignore_index=True)
diagnostics = pd.DataFrame(errors + SCHEMA_WARNINGS, columns=['Раздел', 'Ошибка'])

sheets = {'Сводка': summary, 'Кампании': campaign_result, 'Товары': product_result}
if not phrase_result.empty:
    sheets['Ключи'] = phrase_result
if not funnel_result.empty:
    sheets['Воронка'] = funnel_result
if not econ.empty:
    sheets['Экономика'] = econ
if not diagnostics.empty:
    sheets['Диагностика'] = diagnostics
write_excel(out, sheets)
print(f'В отчёте: кампаний {len(campaign_result)}, товаров {len(product_result)}, ключей {len(phrase_result)}, '
      f'строк воронки {len(funnel_result)}, SKU с экономикой {len(econ)}')


In [22]:
#@title Скачать Excel
from pathlib import Path
from google.colab import files
if 'out' not in globals() or not Path(out).exists():
    raise RuntimeError('Excel ещё не создан. Выполните ячейку «Запуск выгрузки и сборка отчёта» и проверьте лист/сообщения диагностики.')
files.download(out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Как читать экономику

**Маржа до рекламы** по SKU считается на одну проданную штуку:

```
маржа = средняя цена продажи
      − комиссия Ozon (% от цены, по схеме FBO/FBS)
      − логистика (среднее магистрали + последняя миля, из v5/product/info/prices)
      − эквайринг
      − налог (TAX_RATE_PCT от цены)
      − себестоимость (из cost_prices.csv, иначе DEFAULT_COST_SHARE_PCT от цены)
      − EXTRA_COST_PER_UNIT
```

- **Предельный ДРР, %** = маржа ÷ цена × 100 — ДРР, при котором реклама крутится ровно в ноль. Если фактический ДРР ниже — реклама приносит прибыль, выше — съедает её.
- **Предельный CPO, ₽** = маржа — максимальная цена заказа, при которой заказ не убыточен.
- **Запас ДРР, п.п.** = предельный − фактический. Вердикт: запас > `BREAKEVEN_SAFETY_PP` — «В плюс», в пределах ± — «Около нуля», ниже — «В минус».
- **Прибыль с рекламы, ₽** = маржа × заказы с рекламы − расход на рекламу.
- В листе «Кампании» предельный ДРР кампании взвешен по выручке SKU этой кампании; «Покрытие экономикой %» показывает, по какой доле рекламной выручки известна маржа.

## Примечания
- Денежные значения ставок и бюджетов CPC, полученные из настроек кампаний, переводятся из миллионных долей рубля в рубли.
- Средняя цена продажи берётся из рекламной статистики (выручка ÷ заказы); если заказов не было — из текущей цены карточки.
- Логистика и эквайринг — оценки Ozon для текущей цены товара (v5/product/info/prices); фактические удержания смотрите в финансовых отчётах.
- Возвраты и невыкупы в маржу не включены — при заметной доле возвратов фактическая маржа ниже расчётной.
- Полная воронка (переходы в карточку, корзины, выкупы) в `/v1/analytics/data` доступна на подписке Premium Plus; без неё лист «Воронка» содержит только заказы и выручку.
- Асинхронные отчёты Ozon могут возвращаться как JSON, CSV или ZIP — обработчик поддерживает все три варианта.
- Если Ozon изменит название поля или временно не отдаст необязательный раздел, подробность появится на листе **Диагностика**.
